# AprendIA - Demo en Google Colab

Esta demo valida el concepto central de **AprendIA**: un asistente educativo para ninos de basica primaria en zonas rurales de Colombia, pensado para funcionar sin internet y responder solo con material escolar precargado.

La demo no usa APIs externas, no descarga modelos y no usa conocimiento general abierto. El objetivo es que sea facil de explicar en una exposicion.

## Flujo de la demo

1. Se carga material escolar de ejemplo dentro del notebook.
2. El usuario hace una pregunta.
3. Se aplica un filtro basico de seguridad.
4. Se buscan fragmentos relevantes en el material escolar.
5. Se responde usando solo los fragmentos encontrados.
6. Si no hay informacion suficiente, AprendIA responde que no encontro la respuesta en el material escolar.

In [ ]:
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass
from typing import Iterable


## 1. Material escolar precargado

En la app real este contenido estaria empaquetado como una base de conocimiento local, cerrada y de solo lectura. En esta demo lo representamos con una lista de fragmentos validados.

In [ ]:
@dataclass(frozen=True)
class SchoolMaterial:
    id: str
    grade: str
    subject: str
    unit: str
    title: str
    content: str


SCHOOL_MATERIALS = [
    SchoolMaterial(
        id='cn-plantas-01',
        grade='Primaria',
        subject='Ciencias Naturales',
        unit='Las plantas',
        title='La fotosintesis',
        content=(
            'La fotosintesis es el proceso por el cual las plantas fabrican su alimento. '
            'Para hacerlo usan la luz del sol, el agua que toman por las raices y el dioxido de carbono del aire. '
            'Gracias a la fotosintesis, las plantas crecen y producen oxigeno.'
        ),
    ),
    SchoolMaterial(
        id='cn-plantas-02',
        grade='Primaria',
        subject='Ciencias Naturales',
        unit='Las plantas',
        title='Partes de la planta',
        content=(
            'Las plantas tienen varias partes. Las raices sirven para sostener la planta y absorber agua y minerales del suelo. '
            'El tallo transporta el agua hacia las hojas. Las hojas ayudan a fabricar alimento. '
            'La flor puede formar frutos y semillas.'
        ),
    ),
    SchoolMaterial(
        id='mat-suma-01',
        grade='Primaria',
        subject='Matematicas',
        unit='Operaciones basicas',
        title='La suma',
        content=(
            'La suma es una operacion matematica que sirve para juntar cantidades. '
            'Por ejemplo, si tienes 2 mangos y luego recibes 3 mangos mas, en total tienes 5 mangos. '
            'Se puede escribir como 2 + 3 = 5.'
        ),
    ),
    SchoolMaterial(
        id='mat-resta-01',
        grade='Primaria',
        subject='Matematicas',
        unit='Operaciones basicas',
        title='La resta',
        content=(
            'La resta es una operacion matematica que sirve para quitar o comparar cantidades. '
            'Por ejemplo, si tienes 8 semillas y siembras 3, te quedan 5 semillas. '
            'Se puede escribir como 8 - 3 = 5.'
        ),
    ),
    SchoolMaterial(
        id='soc-rio-01',
        grade='Primaria',
        subject='Ciencias Sociales',
        unit='El paisaje',
        title='Los rios',
        content=(
            'Un rio es una corriente natural de agua que se mueve por la superficie de la tierra. '
            'Los rios pueden servir para llevar agua a plantas, animales y comunidades. '
            'Tambien hacen parte del paisaje y deben cuidarse para evitar la contaminacion.'
        ),
    ),
    SchoolMaterial(
        id='est-eval-01',
        grade='Primaria',
        subject='Habitos de estudio',
        unit='Preparacion de evaluaciones',
        title='Como estudiar para una evaluacion',
        content=(
            'Para preparar una evaluacion puedes repasar poco a poco, leer tus apuntes y practicar con preguntas. '
            'Tambien ayuda explicar el tema con tus propias palabras y pedir apoyo a un adulto o docente cuando tengas dudas. '
            'Dormir bien antes de la evaluacion ayuda a concentrarse mejor.'
        ),
    ),
]

len(SCHOOL_MATERIALS)


## 2. Normalizacion de texto

Convertimos el texto a una forma simple para comparar palabras sin depender de mayusculas o tildes.

In [ ]:
STOPWORDS = {
    'a', 'al', 'algo', 'como', 'con', 'cual', 'cuando', 'de', 'del', 'dime', 'el', 'en',
    'es', 'esa', 'ese', 'esto', 'la', 'las', 'lo', 'los', 'me', 'mi', 'para', 'por',
    'que', 'quien', 'se', 'sirve', 'son', 'su', 'un', 'una', 'y'
}


def normalize_text(text: str) -> str:
    normalized = unicodedata.normalize('NFD', text.lower())
    without_accents = ''.join(char for char in normalized if unicodedata.category(char) != 'Mn')
    return re.sub(r'[^a-z0-9\s]', ' ', without_accents)


def tokenize(text: str) -> list[str]:
    return [word for word in normalize_text(text).split() if len(word) > 2 and word not in STOPWORDS]


tokenize('Que es la fotosintesis y para que sirve?')


## 3. Filtro basico de seguridad

Este filtro no reemplaza una capa de seguridad completa. Solo demuestra que algunas preguntas deben bloquearse antes de consultar o responder.

In [ ]:
UNSAFE_PATTERNS = {
    'trampa': 'No puedo ayudar a hacer trampa. Puedo ayudarte a estudiar paso a paso.',
    'copiar': 'No puedo ayudarte a copiar. Puedo darte una pista para que aprendas.',
    'violencia': 'No puedo ayudar con contenido peligroso. Hablemos de un tema escolar seguro.',
    'arma': 'No puedo ayudar con contenido peligroso. Hablemos de un tema escolar seguro.',
}


def safety_check(question: str) -> tuple[bool, str | None]:
    normalized = normalize_text(question)
    for pattern, message in UNSAFE_PATTERNS.items():
        if pattern in normalized:
            return False, message
    return True, None


safety_check('Como hago trampa en un examen?')


## 4. Busqueda local restringida

La busqueda revisa solo el material precargado. No consulta internet ni fuentes externas.

In [ ]:
@dataclass(frozen=True)
class SearchResult:
    material: SchoolMaterial
    score: int
    matched_terms: list[str]


def search_materials(question: str, limit: int = 2) -> list[SearchResult]:
    query_terms = set(tokenize(question))
    if not query_terms:
        return []

    results: list[SearchResult] = []
    for material in SCHOOL_MATERIALS:
        searchable_text = ' '.join([material.subject, material.unit, material.title, material.content])
        material_terms = set(tokenize(searchable_text))
        matched_terms = sorted(query_terms.intersection(material_terms))
        score = len(matched_terms)
        if score > 0:
            results.append(SearchResult(material=material, score=score, matched_terms=matched_terms))

    return sorted(results, key=lambda result: result.score, reverse=True)[:limit]


search_materials('Que es la fotosintesis?')


## 5. Generador de respuesta restringida

La respuesta se arma solo con fragmentos recuperados. Si no hay resultado suficiente, AprendIA no inventa.

In [ ]:
MIN_SCORE = 1
NOT_FOUND_MESSAGE = 'No encontre esa informacion en tu material escolar.'


def build_answer(question: str, results: Iterable[SearchResult]) -> str:
    usable_results = [result for result in results if result.score >= MIN_SCORE]
    if not usable_results:
        return NOT_FOUND_MESSAGE

    main_result = usable_results[0]
    material = main_result.material
    return (
        'Segun tu material escolar:\n\n'
        f'{material.content}\n\n'
        f'Fuente: {material.subject} - {material.unit} - {material.title}'
    )


def preguntar(question: str) -> str:
    is_safe, safety_message = safety_check(question)
    if not is_safe:
        return safety_message or 'No puedo responder esa pregunta de forma segura.'

    results = search_materials(question)
    return build_answer(question, results)


## 6. Pruebas listas para exposicion

Ejecuta estas preguntas para mostrar cuando AprendIA responde y cuando se niega a inventar.

In [ ]:
demo_questions = [
    'Que es la fotosintesis?',
    'Para que sirven las raices de una planta?',
    'Que es una suma?',
    'Que es un rio?',
    'Como puedo prepararme para una evaluacion?',
    'Que es un agujero negro?',
    'Como hago trampa en un examen?',
]

for demo_question in demo_questions:
    print('=' * 80)
    print('Pregunta:', demo_question)
    print()
    print(preguntar(demo_question))
    print()


## 7. Inspeccion de busqueda

Esta celda ayuda a explicar por que se eligio una fuente.

In [ ]:
def explicar_busqueda(question: str) -> None:
    print('Pregunta:', question)
    print('Terminos usados:', tokenize(question))
    print()
    results = search_materials(question, limit=5)
    if not results:
        print('No hubo coincidencias en el material escolar precargado.')
        return

    for index, result in enumerate(results, start=1):
        material = result.material
        print(f'{index}. {material.subject} / {material.unit} / {material.title}')
        print('   Puntaje:', result.score)
        print('   Coincidencias:', ', '.join(result.matched_terms))


explicar_busqueda('Para que sirven las raices?')


## 8. Demo interactiva

Esta interfaz permite probar AprendIA sin escribir codigo. Escribe una pregunta o usa uno de los ejemplos. La respuesta seguira usando solo el material escolar precargado.

In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display


question_input = widgets.Textarea(
    value='',
    placeholder='Escribe tu pregunta escolar aqui...',
    description='Pregunta:',
    layout=widgets.Layout(width='100%', height='90px'),
)

ask_button = widgets.Button(
    description='Preguntar a AprendIA',
    button_style='success',
    icon='search',
)

clear_button = widgets.Button(
    description='Limpiar',
    button_style='warning',
    icon='trash',
)

response_output = widgets.Output()


def render_answer(question: str) -> None:
    with response_output:
        clear_output()
        clean_question = question.strip()
        if not clean_question:
            display(Markdown('**Escribe una pregunta primero.**'))
            return

        answer = preguntar(clean_question)
        display(Markdown(f'### Pregunta\n{clean_question}'))
        display(Markdown(f'### Respuesta de AprendIA\n{answer}'))


def on_ask_clicked(_button: widgets.Button) -> None:
    render_answer(question_input.value)


def on_clear_clicked(_button: widgets.Button) -> None:
    question_input.value = ''
    with response_output:
        clear_output()


ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

display(Markdown('## AprendIA - Demo interactiva'))
display(Markdown('Escribe una pregunta escolar. AprendIA respondera solo con el material precargado.'))
display(question_input)
display(widgets.HBox([ask_button, clear_button]))
display(response_output)


### Preguntas de ejemplo

Usa estos botones para cargar preguntas rapidamente en la caja de texto.

In [ ]:
example_questions = [
    'Que es la fotosintesis?',
    'Para que sirven las raices de una planta?',
    'Que es una suma?',
    'Como puedo prepararme para una evaluacion?',
    'Que es un agujero negro?',
    'Como hago trampa en un examen?',
]


def make_example_button(question: str) -> widgets.Button:
    button = widgets.Button(
        description=question[:42],
        layout=widgets.Layout(width='100%', margin='2px 0'),
    )

    def on_example_clicked(_button: widgets.Button) -> None:
        question_input.value = question
        render_answer(question)

    button.on_click(on_example_clicked)
    return button


display(widgets.VBox([make_example_button(question) for question in example_questions]))


## 9. Limitaciones y siguientes pasos

Esta demo es intencionalmente minima. Valida el principio mas importante: AprendIA responde solo desde material escolar precargado.

Siguientes pasos tecnicos:

- Cambiar busqueda simple por busqueda hibrida o embeddings locales.
- Empaquetar la base de conocimiento como SQLite de solo lectura o JSON optimizado.
- Crear la app Android con Jetpack Compose.
- Agregar historial local.
- Agregar Text-to-Speech offline.
- Evaluar modelos locales pequenos para Android.
- Mantener siempre la regla: si no esta en el material escolar, no se responde como verdad.